# 02. Modelowanie cen Fixing I
 
**Autor**: Damian Urbański

Notebook realizuje plan z [notebooka 1](./01_data_exploration.ipynb). Każdy przebieg to osobny run w MLflow. 

## Importy i konfiguracja środowiska

In [1]:
import os
import subprocess
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from experiments.da_energy_prices_forecasts.utils.features import (
    END_DATE,
    FEATURES_VERSION,
    START_DATE,
    TARGET,
    make_training_frame,
)
from experiments.da_energy_prices_forecasts.utils.modeling import (
    ALL_COLUMNS,
    MODEL_INFO,
    REGISTERED_MODEL_NAME,
    Spec,
    benchmark_predictions,
    build_ensemble,
    build_estimator,
    compute_metrics,
    default_matrix,
    split_days,
    tune,
    walk_forward,
)
from mlflow import MlflowClient
from mlflow.models import infer_signature
from scipy.stats import linregress
from sklearn.exceptions import ConvergenceWarning
from statsmodels.graphics.tsaplots import plot_acf
from utils.db import EnergyDatabase

load_dotenv()

warnings.filterwarnings("ignore", category=ConvergenceWarning)
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
EXPERIMENT_NAME = "da_price_fixing1_v1"
energy_db = EnergyDatabase()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print("MLflow:", MLFLOW_TRACKING_URI, "| eksperyment:", EXPERIMENT_NAME)

MLflow: http://100.115.91.9:5000 | eksperyment: da_price_fixing1_v1


## Przygotowanie danych

In [2]:
raw_data = energy_db.fetch_by_datetime("forecasts_and_real", 
                                   columns=["datetime_utc", "pv_fcst_pse", "wind_fcst_pse", "demand_fcst_pse", "exchange_fcst_pse", "resload_fcst_pse", "fixing_1_pln_mwh",
                                            "pv_actual_pse", "wind_actual_pse", "demand_actual_pse", "exchange_actual_pse", "resload_actual_pse",], 
                                   start_date = START_DATE, 
                                   end_date = END_DATE)
raw_data.set_index("datetime_utc", inplace=True)
required = [TARGET, "demand_fcst_pse", "pv_fcst_pse", "wind_fcst_pse", "exchange_fcst_pse"]
print(raw_data.index.min(), "->", raw_data.index.max(), "|", len(raw_data), "godzin")

2026-08-01 00:00:00+00:00 -> 2026-10-06 23:00:00+00:00 | 1608 godzin


In [3]:
data = make_training_frame(raw_data, dropna = True)
y = data[TARGET]

In [4]:
days = sorted(data["delivery_date"].unique())

# tune_days - dni do treningu modelu, wf_days - dni do backtestu, holdout_days - dni do potwierdzenia wyboru najważneijszych modeli
tune_days, wf_days, holdout_days = split_days(days)

pd.DataFrame(
    {
        "etap": ["tune", "backtest", "hodlout"],
        "start": [tune_days[0], wf_days[0], holdout_days[0]],
        "end": [tune_days[-1], wf_days[-1], holdout_days[-1]],
        "period (days)": [len(tune_days), len(wf_days), len(holdout_days)]
    }
)

,etap,start,end,period (days)
0,tune,2026-08-02,2026-09-05,35
1,backtest,2026-09-06,2026-09-30,25
2,hodlout,2026-10-01,2026-10-07,7


## Narzędzia do MLflow

In [5]:
METRIC_COLS = ["mae", "rmse", "wape", "bias", "mae_peak_16_19", "daily_mae_median", "daily_mae_iqr"]

COMMON_PARAMS = {
    "date_start": str(raw_data.index.min()),
    "date_end": str(raw_data.index.max()),
    "n_days_total": len(days),
    "n_days_tune": len(tune_days),
    "n_days_walk_forward": len(wf_days),
    "n_days_holdout": len(holdout_days),
    "inputs": "forecasts"
}

def _unwrap(v):
    return v.item() if hasattr(v, "item") else v

def plot_forecast(y_true, y_pred, title, last_days =7):
    idx = y_pred.index[-24*last_days:]
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.plot(idx, y_true.loc[idx], label = "Fixing I real")
    ax.plot(idx, y_pred.loc[idx], label= "Forecast")
    ax.set_title(title)
    ax.set_ylabel("PLN/MWh")
    ax.legend()
    fig.tight_layout()
    return fig

def plot_residuals(y_true, y_pred, title):
    idx = y_pred.index
    y_true = y_true.loc[idx].copy()
    residuos = y_true - y_pred
    fig, ax = plt.subplots(figsize=(10,6))
    ax.scatter(y_pred, residuos, alpha=0.7)
    ax.axhline(y=0, color='r', linestyle = '--')
    ax.set_xlabel("Wartości prognozowane")
    ax.set_ylabel("Residua")
    ax.set_title(title)
    slope, intercept, r_value, p_value, std_err = linregress(y_pred, residuos)
    x_line = np.array([min(y_pred), max(y_pred)])
    y_line = slope * x_line + intercept
    p_str = "p < 0.0001" if p_value < 0.0001 else f"p = {p_value:.4f}"
    ax.plot(x_line, y_line, 'g--', linewidth =2, label=f"Trend (p-value: {p_str})")
    ax.legend()
    plt.tight_layout()
    return fig

def plot_residuals_acf(y_true, y_pred, title, lags = 48):
    idx = y_pred.index
    y_true = y_true.loc[idx].copy()
    residous = y_true - y_pred
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_acf(residous, lags = lags, ax = ax, zero=False)
    ax.set_title(title)
    ax.set_xlabel("Opóźnienie (h)")
    ax.set_ylabel("Współczynnik autokorelacji")
    plt.tight_layout()
    return fig

def plot_hourly_residuals(y_true, y_pred, title, local_tz="Europe/Warsaw"):
    idx = y_pred.index
    y_true = y_true.loc[idx].copy()
    res = pd.DataFrame(
        {
            "res": y_true - y_pred,
            "hour": y_pred.index.tz_convert(local_tz).hour if hasattr(y_pred.index, "tz_convert") else y_pred.index.hour
        }
    )

    fig, ax = plt.subplots(figsize=(11,5))
    res.boxplot(column="res", by="hour", ax = ax, grid = False, showfliers=False)
    ax.axhline(0, color = "r", linestyle = "--")
    ax.set_title("Rozkład residuów wg godzin doby")
    ax.set_xlabel("Godzina doby")
    ax.set_ylabel("Residua")
    plt.suptitle(title)
    plt.tight_layout()
    return fig

def return_all_plots(y_true, y_pred, title):
    fig_forecasts = plot_forecast(y_true, y_pred, title)
    fig_hourly_residuals = plot_hourly_residuals(y_true, y_pred, title)
    fig_residuals_acf = plot_residuals_acf(y_true, y_pred, title)
    fig_residuals = plot_residuals(y_true, y_pred, title)

    return fig_forecasts, fig_hourly_residuals, fig_residuals_acf, fig_residuals

def log_run(run_name, stage, tags, params, metrics, preds, best_params=None):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({"stage": stage, **tags})
        mlflow.log_params({**COMMON_PARAMS, **{k: _unwrap(v) for k, v in params.items()}})
        mlflow.log_metrics({k: v for k, v in metrics.items() if not np.isnan(v)})
        mlflow.log_text(preds.rename("pred").to_csv(), "predictions.csv")
        if best_params is not None:
            mlflow.log_dict({k: _unwrap(v) for k, v in best_params.items()}, "best_params.json")

        fig_forecasts, fig_hourly_residuals, fig_residuals_acf, fig_residuals = return_all_plots(y, preds, f"{run_name} ({stage})")

        for fig, name in (
            (fig_forecasts, "forecast_vs_actual.png"),
            (fig_hourly_residuals, "hourly_residuals.png"),
            (fig_residuals_acf, "acf_residuals.png"),
            (fig_residuals, "residuals.png")
        ):
            mlflow.log_figure(fig, name)
            plt.close(fig)

        return run.info.run_id

def table(results):
    rows = {n: {c: r['metrics'].get(c) for c in METRIC_COLS} for n, r in results.items()}
    return pd.DataFrame(rows).T.sort_values("mae").round(3)

## Benchmarki

In [6]:
BENCH = {}

bench_wf = benchmark_predictions(data, wf_days)
b0_wf = bench_wf["B0_persistance_t24"]
for name, preds in bench_wf.items():
    m = compute_metrics(y, preds, b0_wf)
    BENCH[name]={"metrics":m, "preds":preds}
    log_run(name, "walk_forward", {"model_id": name.split("_")[0], "kind":"benchmark"},
            {"model_id": name, "feature_set": "-", "target": "-", "loss": "-"}, m, preds.dropna())
table(BENCH)

🏃 View run B0_persistance_t24 at: http://100.115.91.9:5000/#/experiments/2/runs/80962d06322148c090f2295f43984879
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
🏃 View run B1_persistance_typeday at: http://100.115.91.9:5000/#/experiments/2/runs/f5c849eff46c45cf98601db0fe292ff6
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
🏃 View run B2_profile_7d at: http://100.115.91.9:5000/#/experiments/2/runs/210a48e2392d4f2a90d89c1b4d34d78c
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
🏃 View run B2_profile_14d at: http://100.115.91.9:5000/#/experiments/2/runs/16ffc6f297d24746b4cdd6dd9d5e732a
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2


,mae,rmse,wape,bias,mae_peak_16_19,daily_mae_median,daily_mae_iqr
B0_persistance_t24,166.805,241.749,0.246,-12.292,258.851,155.715,124.615
B2_profile_14d,172.012,239.274,0.253,-20.850,283.482,130.940,110.523
B2_profile_7d,189.386,248.335,0.279,-8.074,317.360,166.385,105.991
B1_persistance_typeday,194.927,271.636,0.287,-15.667,261.376,157.208,210.418


## Trenowanie modeli

In [7]:
RESULTS = {}

def with_params(est, params):
    est.set_params(**params)
    return est

def evaluate(name, make_model, days_, stage, b0, params, tags, best=None):
    preds = walk_forward(make_model(), data, days_)
    m = compute_metrics(y, preds, b0)
    run_name = name if stage == "walk_forward" else f"HOLDOUT_{name}"
    run_id = log_run(run_name, stage, tags, params, m, preds, best_params=best)
    return {"metrics": m, "preds": preds, "run_id":run_id}

def run_spec(spec):
    best, cv_mae = tune(spec, data, tune_days)
    desc, family, loss, _ = MODEL_INFO[spec.model_id]
    flat_params = {k.split("__")[-1]: v for k, v in best.items()}
    params = {
        "model_id": spec.model_id,
        "model":desc,
        "feature_set": spec.fset,
        "target": spec.target,
        "loss": loss, 
        "cv_mae_tuning": round(cv_mae, 3),
        **flat_params
    }

    make = lambda: with_params(build_estimator(spec), best)

    evaluation = evaluate(
        spec.name,
        make,
        wf_days,
        "walk_forward",
        b0_wf,
        params,
        {
            "model_id": spec.model_id,
            "feature_set": spec.fset,
            "kind": "model"
        },
        best = best
    )
    RESULTS[spec.name] = {
        **evaluation,
        "spec": spec,
        "best": best,
        "cv_mae": cv_mae,
        "make": make,
        "family": family,
        "model_id": spec.model_id
    }

    metrics = evaluation["metrics"]
    print(f"{spec.name:12s} cv_mae={cv_mae:7.2f}  wf_mae={metrics['mae']:7.2f}  rMAE={metrics['rmae']:.3f}")


for spec in default_matrix():
    run_spec(spec)

results_table = table(RESULTS)

🏃 View run M1_Z0 at: http://100.115.91.9:5000/#/experiments/2/runs/a0c0772e228c4c87868ceb293d1fbd91
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
M1_Z0        cv_mae= 112.57  wf_mae= 142.94  rMAE=0.857
🏃 View run M1_Z1 at: http://100.115.91.9:5000/#/experiments/2/runs/bbcfe9cabf1849e7ab689752243151fb
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
M1_Z1        cv_mae=  99.43  wf_mae= 106.86  rMAE=0.641
🏃 View run M1_Z2 at: http://100.115.91.9:5000/#/experiments/2/runs/5188ccb4591a409a86e6d695aaf83755
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
M1_Z2        cv_mae=  90.06  wf_mae=  98.10  rMAE=0.588
🏃 View run M1_Z3 at: http://100.115.91.9:5000/#/experiments/2/runs/0eee0a8eee07473cbe96dacac37c55cc
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
M1_Z3        cv_mae=  88.70  wf_mae=  97.98  rMAE=0.587
🏃 View run M2_Z1 at: http://100.115.91.9:5000/#/experiments/2/runs/acf37b0c2e1c4c7186545e8049444376
🧪 View experiment at: ht

## Transformacja zmiennej celu dla najlepszego modelu liniowego i najlepszego LightGBM

In [8]:
best_linear = next((n for n in results_table.index if RESULTS[n]["family"] == "linear"), None)
best_gbm = next((n for n in results_table.index if RESULTS[n]["model_id"] in ("M3", "M4")), None)

for name in (best_linear, best_gbm):
    specs = RESULTS[name]["spec"]
    run_spec(Spec(specs.model_id, specs.fset, "asinh"))

table(RESULTS)

🏃 View run M1_Z3_asinh at: http://100.115.91.9:5000/#/experiments/2/runs/559d2ce894aa453d9eb19f35ac96edfb
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
M1_Z3_asinh  cv_mae= 101.06  wf_mae= 112.28  rMAE=0.673
🏃 View run M3_Z2_asinh at: http://100.115.91.9:5000/#/experiments/2/runs/84b8c73f15c24fc2ad11f56111345379
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
M3_Z2_asinh  cv_mae=  88.58  wf_mae=  95.24  rMAE=0.571


,mae,rmse,wape,bias,mae_peak_16_19,daily_mae_median,daily_mae_iqr
M3_Z2,93.482,155.037,0.138,-40.749,179.226,75.313,37.739
M3_Z3,93.591,161.638,0.138,-42.094,185.966,73.646,29.529
M5_Z3,94.293,151.470,0.139,-34.451,169.573,82.830,25.334
M3_Z2_asinh,95.239,153.319,0.140,-35.680,174.246,80.777,30.187
M4_Z3,96.828,159.167,0.143,-39.099,184.748,80.900,29.943
M1_Z3,97.982,152.334,0.144,-41.911,172.389,81.615,26.636
M2_Z2,98.082,154.490,0.144,-46.577,175.135,77.905,31.539
M1_Z2,98.100,149.382,0.145,-38.897,169.874,79.751,25.843
M4_Z1,102.579,166.526,0.151,-31.536,186.617,88.749,37.757
M2_Z1,105.001,167.434,0.155,-50.557,201.490,91.406,33.144


## Model ensamble

In [9]:
specs = [RESULTS[best_linear]["spec"], RESULTS[best_gbm]["spec"]]
bests = [RESULTS[best_linear]["best"], RESULTS[best_gbm]["best"]]

make_ens = lambda: build_ensemble(specs, bests)
out = evaluate("M6_ensemble", make_ens, wf_days, "walk_forward", b0_wf,
               {"model_id": "M6", "model": "Voting (srednia)", "feature_set": "mieszane", "target": "mieszane",
                "loss": "n/d", "members": f"{best_linear} + {best_gbm}"},
               {"model_id": "M6", "feature_set": "mieszane", "kind": "model"})
RESULTS["M6_ensemble"] = {**out, "spec": None, "best": None, "cv_mae": None, "make": make_ens,
                          "family": "ensemble", "model_id": "M6", "complexity": 5}

table({**BENCH, **RESULTS})

🏃 View run M6_ensemble at: http://100.115.91.9:5000/#/experiments/2/runs/b12307de42c044e086f054c71057d41f
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2


,mae,rmse,wape,bias,mae_peak_16_19,daily_mae_median,daily_mae_iqr
M6_ensemble,92.874,150.641,0.137,-41.330,173.610,76.458,26.386
M3_Z2,93.482,155.037,0.138,-40.749,179.226,75.313,37.739
M3_Z3,93.591,161.638,0.138,-42.094,185.966,73.646,29.529
M5_Z3,94.293,151.470,0.139,-34.451,169.573,82.830,25.334
M3_Z2_asinh,95.239,153.319,0.140,-35.680,174.246,80.777,30.187
M4_Z3,96.828,159.167,0.143,-39.099,184.748,80.900,29.943
M1_Z3,97.982,152.334,0.144,-41.911,172.389,81.615,26.636
M2_Z2,98.082,154.490,0.144,-46.577,175.135,77.905,31.539
M1_Z2,98.100,149.382,0.145,-38.897,169.874,79.751,25.843
M4_Z1,102.579,166.526,0.151,-31.536,186.617,88.749,37.757


## Wybór najlepszych modeli

In [10]:
def daily_mae(name):
    p = RESULTS[name]["preds"]
    e = (p - y.loc[p.index]).abs()
    return e.groupby(p.index.tz_convert("Europe/Warsaw").date).mean()

d = daily_mae("M1_Z2") - daily_mae("M3_Z2")
print(d.mean(), d.std() / np.sqrt(len(d)))   # średnia różnica i jej błąd standardowy

4.617724754487998 3.2424169276495705


**Wnioski z przeprowadzonych eksperymentów**

1. Benchmark z uwzględnieniem typu dnia `B1_persistance_typeday` wychodzi gorzej niż prosty `B0_persistance_t24` (MAE 194,9 vs 166,8). Prawdopodobnie w tym przypadku bardziej szkodzi tak długi odstęp (168 h) niż pomaga efekt dnia tygodnia. Możliwe również, że kilka pojedynczych dni zawyża średnią.
2. Wszystkie modele uczone osiągają lepsze MAE niż najlepszy benchmark (B0: 166,8; najlepszy model: 92,9, czyli ok. 44% mniej).
3. Wybór zestawu cech wyraźnie wpływa na wynik. Zestaw `Z0` (kalendarz i lag cenowy) nie wystarcza do odwzorowania cen: poprawia benchmark B0 zaledwie o 10–14% (MAE 143–150 wobec 166,8). Dodanie prognoz fundamentalnych (`Z1`) obniża MAE tego samego modelu o ok. 25–29% (M1: 142,9 → 106,9; M3: 149,7 → 105,9). Kolejny skok daje przejście na `Z2` lub `Z3`: w każdej z pięciu rodzin modeli są one lepsze od `Z1` o 6–14% (np. M1: 106,9 → 98,1; M3: 105,9 → 93,5; M5: 109,4 → 94,3). Między `Z2` a `Z3` nie widać różnicy (M1: 98,1 vs 98,0; M3: 93,5 vs 93,6), więc dodanie `res_share` nic nie wnosi. Zastrzeżenie: w modelach liniowych splajn na `resload` występuje tylko w `Z2` i `Z3`, więc część przewagi nad `Z1` może wynikać ze splajnu, a nie z samego `resload`. W modelach drzewiastych splajnu nie ma.
4. Dziewięć najlepszych modeli mieści się w przedziale MAE 92,9–98,1 (ok. 5%). Dzienne MAE ma rozstęp międzykwartylowy rzędu 25–40. W połączeniu z niewielką liczbą dób testowych (25) różnica MAE jest prawdopodobnie w granicach szumu: w teście parami dla M1_Z2 i M3_Z2 średnia różnica dziennego MAE wynosi 4,6 PLN/MWh przy błędzie standardowym 3,2, czyli poniżej dwóch SE.
5. Wszystkie modele zaniżają ceny (bias od -31 do -65), natomiast benchmarki mają mniejszy co do wartości bezwzględnej bias (-8 do -21). Hipoteza: ceny rosły od sierpnia do października, a model uczony na rosnącym oknie uczył się niższych poziomów cen z przeszłości, podczas gdy benchmarki śledziły poziom z poprzedniej doby lub poprzednich dni.
6. Transformacja zmiennej celu wyraźnie pogorszyła model Ridge'a (MAE 112,3 vs 98,0), a u LightGBM nie przyniosła mierzalnej poprawy (95,2 vs 93,5). Prawdopodobnie `asinh` ściska piki, a funkcja odwrotna zaniża prognozy dokładnie w tych miejscach (u Ridge'a bias spadł z -41,9 do -64,7).
7. Połączenie prognoz w model zespołowy (`M6_ensemble`) pozwoliło uzyskać najlepszy wynik w całym zestawieniu: MAE spadło do 92,87 PLN/MWh, a RMSE do 150,64 PLN/MWh. Uśrednienie predykcji zróżnicowanych architektur skutecznie stłumiło wariancję błędu (IQR na poziomie 26,39) oraz ograniczyło błędy w godzinach szczytowych (MAE w oknie 16–19 spadło do 173,61 względem 179,23 w M3_Z2).
8. Analiza wykresów modeli wykazała:
    - Godzina 19 (czasu lokalnego) jest najgorszą godziną we wszystkich modelach. W najlepszych modelach mediana residuów wynosi ok. 140–180 PLN/MWh, a IQR ok. 350. Modele konsekwentnie i systematycznie niedoszacowują tę godzinę. Drugie skupisko błędów to godziny 6–7. Największe błędy występują więc w godzinach, gdy zmienia się struktura podaży (wieczorem PV zanika, rano startuje).
    - Autokorelacja błędów sugeruje, że residua nie są szumem białym: jeśli model pomylił się o `x` w danej godzinie, to godzinę później myli się podobnie. Model nie widzi więc czegoś, co zmienia się wolno. Mogą to być zmienne zewnętrzne (np. ceny paliw i $CO_2$) albo informacja o dynamice, która jest już w danych, ale nie została podana modelowi (np. tempo zmiany `resload` względem poprzedniej godziny). Dodatnia autokorelacja przy lagu 24 prawdopodobnie wynika z trendu wzrostowego cen w badanym okresie.

### Wybór najlepszych modeli

Z racji, że różnica MAE wśród najlepszych modeli jest w granicach szumu (np. różnica 4,6 PLN przy błędzie standardowym 3,2, czyli poniżej 2SE), ostateczny wybór oparto na **RMSE** oraz **dziennym IQR**. Kryteria te silniej karzą krytyczne, duże błędy (jak trudna godzina 19 z medianą residuów 140–180 PLN i IQR rzędu 350).

Mimo że model `M1_Z2` osiąga marginalnie wyższe MAE, ze względów bezpieczeństwa (zarządzanie ryzykiem) wskazano dwa czołowe modele:
- **`M6_ensemble`** (model zespołowy)
- **`M1_Z2`** (model liniowy ze splajnem na `resload`)

In [11]:
table(RESULTS).sort_values(by=["rmse", "daily_mae_iqr"]).head(2)

,mae,rmse,wape,bias,mae_peak_16_19,daily_mae_median,daily_mae_iqr
M1_Z2,98.100,149.382,0.145,-38.897,169.874,79.751,25.843
M6_ensemble,92.874,150.641,0.137,-41.330,173.610,76.458,26.386


### Holdout

In [12]:
finalists = ['M6_ensemble', 'M1_Z2']

In [13]:
bench_ho = benchmark_predictions(data, holdout_days)
b0_ho = bench_ho["B0_persistance_t24"]
HOLDOUT = {}

for name, preds in bench_ho.items():
    m = compute_metrics(y, preds, b0_ho)
    HOLDOUT[name] = {"metrics": m, "preds": preds}
    log_run(
        f"HOLDOUT_{name}",
        "holdout",
        {
            "model_id": name.split("_")[0],
            "kind": "benchmark"
        },
        {
            "model_id": name,
            "feature_set": "-",
            "target": "-",
            "loss": "-"
        },
        m,
        preds
    )

for name in finalists:
    res = RESULTS[name]
    params = {"model_id": res["model_id"], "run_wf": name}
    HOLDOUT[name] = evaluate(
        name,
        res["make"],
        holdout_days,
        "holdout",
        b0_ho,
        params,
        {
            "model_id": res["model_id"],
            "kind": "model"
        },
        best = res["best"]
    )
table(HOLDOUT)

🏃 View run HOLDOUT_B0_persistance_t24 at: http://100.115.91.9:5000/#/experiments/2/runs/77ec594d6c2045b597fdd8dfddd05e03
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
🏃 View run HOLDOUT_B1_persistance_typeday at: http://100.115.91.9:5000/#/experiments/2/runs/b0bb5eca2309423f8e34f1bcf7b5713d
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
🏃 View run HOLDOUT_B2_profile_7d at: http://100.115.91.9:5000/#/experiments/2/runs/098ff9bf50c845a9bb84f7bc3ac2aea9
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
🏃 View run HOLDOUT_B2_profile_14d at: http://100.115.91.9:5000/#/experiments/2/runs/c7b0ab5d6bfe45e0bdf9b4a146a08e2c
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
🏃 View run HOLDOUT_M6_ensemble at: http://100.115.91.9:5000/#/experiments/2/runs/8f4a73b1896d4bd3b86df51df3422531
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
🏃 View run HOLDOUT_M1_Z2 at: http://100.115.91.9:5000/#/experiments/2/runs/d4f369d0cb1447b88

,mae,rmse,wape,bias,mae_peak_16_19,daily_mae_median,daily_mae_iqr
M6_ensemble,70.837,87.671,0.105,-39.002,77.067,73.122,24.011
M1_Z2,84.314,100.278,0.125,-46.335,81.236,82.545,15.635
B2_profile_7d,97.742,132.499,0.144,-52.377,177.109,89.857,44.276
B2_profile_14d,102.087,127.134,0.151,-42.209,170.918,105.856,43.775
B1_persistance_typeday,108.841,152.213,0.161,-84.267,184.903,121.565,69.251
B0_persistance_t24,118.272,158.816,0.175,-30.064,211.504,130.251,75.152


## Model końcowy: trening na wszystkich danych i rejestracja modelu

In [14]:
FINAL = finalists[0] #model M6_ensemble
MODEL_ALIAS = "champion"

res = RESULTS[FINAL]

X_all = data[ALL_COLUMNS]
final_model = res["make"]().fit(X_all, y)
sample = X_all.head(50)
git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()


with mlflow.start_run(run_name=f"FINAL_{FINAL}") as run:
    mlflow.set_tags({
        "stage": "final",
        "model_id": res["model_id"],
        "inputs": "forecasts",
        "git_sha": git_sha,
        "features_version": FEATURES_VERSION,
    })
    mlflow.log_params({**COMMON_PARAMS, "final_run": FINAL, "n_rows_train": len(X_all)})
    mlflow.log_metrics({f"wf_{k}": v for k, v in res["metrics"].items() if not np.isnan(v)})
    mlflow.log_metrics({f"ho_{k}": v for k, v in HOLDOUT[FINAL]["metrics"].items() if not np.isnan(v)})
    mlflow.sklearn.log_model(
        final_model,
        name="model",
        signature = infer_signature(sample, final_model.predict(sample)),
        input_example = sample.head(3),
        registered_model_name=REGISTERED_MODEL_NAME,
        serialization_format="cloudpickle",
    )
    final_run_id = run.info.run_id

client = MlflowClient()
versions = [v for v in client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'") if v.run_id == final_run_id]
version = max(versions, key=lambda v: int(v.version)).version
client.set_registered_model_alias(REGISTERED_MODEL_NAME, MODEL_ALIAS, version)
client.set_model_version_tag(REGISTERED_MODEL_NAME, version, "holdout_mae", f"{HOLDOUT[FINAL]['metrics']['mae']:.2f}")
print(f"Zarejestrowano {REGISTERED_MODEL_NAME} v{version} (alias: {MODEL_ALIAS}), run {final_run_id}")

2026/10/13 23:48:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/13 23:48:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'da_price_fixing1'.
2026/10/13 23:48:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: da_price_fixing1, version 1
Created version '1' of model 'da_price_fixing1'.


🏃 View run FINAL_M6_ensemble at: http://100.115.91.9:5000/#/experiments/2/runs/26b1d788a7ac4d4d9c5fda8d884a7eb8
🧪 View experiment at: http://100.115.91.9:5000/#/experiments/2
Zarejestrowano da_price_fixing1 v1 (alias: champion), run 26b1d788a7ac4d4d9c5fda8d884a7eb8
